In [1]:
import os
import sys

import geopandas as gpd
import pandas as pd
import yaml

# on se place à la racine du projet
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())

# paramètres communs du projet
with open("config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

from src.variables import build_communes, build_gares, build_voyageurs_communes

# les tables propres de chaque base
contours = gpd.read_parquet("data/clean/contours_communes.parquet")
population = pd.read_parquet("data/clean/population.parquet")
navettes = pd.read_parquet("data/clean/navettes.parquet")
parc = pd.read_parquet("data/clean/parc_auto.parquet")
# table SNCF de Mathilde : une ligne par gare et par année
sncf = pd.read_parquet("data/clean/sncf-merged.parquet")

In [2]:
gares = build_gares(sncf, contours)
voyageurs = build_voyageurs_communes(sncf, gares, config["annee_debut"], config["annee_fin"])

# la grande table : une ligne par commune
communes = build_communes(contours, population, navettes, parc, voyageurs)
print(communes.shape)

methode
code SNCF                 2730
arrondissement              49
point dans le polygone      26
Name: count, dtype: int64
gares sans commune : 0
population : 34746 communes, 0 sans correspondance (0.00 %)
navettes : 34746 communes, 0 sans correspondance (0.00 %)
parc automobile : 34746 communes, 4 sans correspondance (0.01 %)
gares et voyageurs : 34746 communes, 32295 sans correspondance (92.95 %)
(34746, 35)


In [3]:
communes.to_parquet("data/clean/communes.parquet")
gares.to_parquet("data/clean/gares.parquet")